# JMI catalyst screening: Catalysis-Hub export and data audit

Builds the surface-level hydrogen-evolution dataset used in *False-Discovery-Controlled
Screening of Bimetallic Alloy Surfaces for Hydrogen Evolution* and reproduces Tables 1 and 2.

1. Export publication `MamunHighT2019` (reactant `H2gas`, product `Hstar`) from Catalysis-Hub.
2. Download the structure-type metadata used for the cross-check.
3. De-duplicate, aggregate configurations to surfaces, compute ΔG_H and the feasibility check.

Inputs stay in `MyDrive/Datasets/`; large working files go to `/content`; the small result
files are saved to `MyDrive/Outputs/JMI_Catalyst_Screening/`.

In [ ]:
# Setup: paths and Drive
import os, sys, json, time, hashlib
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    MYDRIVE = Path("/content/drive/MyDrive")
    WORK = Path("/content/jmi_catalyst")
else:                                   # local run
    MYDRIVE = Path(os.environ.get("MYDRIVE", str(Path.home() / "MyDrive")))
    WORK = Path(os.environ.get("JMI_WORK", "/tmp/jmi_catalyst"))

DATA_DIR = MYDRIVE / "Datasets" / "CatalysisHub_MamunHighT2019_H"
OUT_DIR = MYDRIVE / "Outputs" / "JMI_Catalyst_Screening"
EXPORT_CSV = DATA_DIR / "mamun2019_H_reactions.csv"
EXPECTED_SHA256 = "2545b6912f33835e51506ec42c5362c77c5f1cd34912e0a37f95e2ba76d7ac99"
for p in (DATA_DIR, OUT_DIR, WORK):
    p.mkdir(parents=True, exist_ok=True)
print("Dataset folder:", DATA_DIR)
print("Results folder:", OUT_DIR)
print("Working folder:", WORK)

In [ ]:
# Step 1: export from Catalysis-Hub (skipped if the export is already in Drive)
import requests, pandas as pd, getpass

def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

API = "https://api-catalysis-hub.slac.stanford.edu/graphql"
Q = """query($after: String) {
  reactions(pubId: "MamunHighT2019", reactants: "H2gas", products: "Hstar",
            order: "id", first: 200, after: $after) {
    totalCount
    pageInfo { hasNextPage endCursor }
    edges { node { id Equation sites coverages reactants products facet
                   reactionEnergy dftCode dftFunctional
                   surfaceComposition chemicalComposition
                   reactionSystems { name aseId } } }
  }
}"""

if EXPORT_CSV.exists():
    print(f"Export found: {EXPORT_CSV}")
else:
    key = getpass.getpass("Catalysis-Hub API key (hidden): ")
    rows, seen, after, page, total = [], set(), "", 0, None
    while True:
        for attempt in range(5):
            r = requests.post(API, json={"query": Q, "variables": {"after": after}},
                              headers={"X-API-Key": key}, timeout=60)
            if r.status_code == 200 and "errors" not in r.json():
                break
            print(f"  retry {attempt+1}: HTTP {r.status_code} {r.text[:150]}")
            time.sleep(5 * (attempt + 1))
        else:
            raise RuntimeError("Export stopped: repeated API errors (see messages above).")
        data = r.json()["data"]["reactions"]
        total = data["totalCount"]
        new = 0
        for e in data["edges"]:
            n = e["node"]
            if n["id"] in seen:
                continue
            seen.add(n["id"]); new += 1
            n["reactionSystems"] = json.dumps(n["reactionSystems"])
            rows.append(n)
        page += 1
        print(f"page {page}: +{new} new, {len(rows)} / {total} distinct records")
        if not data["pageInfo"]["hasNextPage"]:
            break
        after = data["pageInfo"]["endCursor"]
    del key
    if len(rows) != total:
        raise RuntimeError(f"Incomplete export: {len(rows)} distinct of {total}. Not saved.")
    pd.DataFrame(rows).to_csv(EXPORT_CSV, index=False)
    print(f"Saved {len(rows)} distinct records to {EXPORT_CSV}")

digest = sha256(EXPORT_CSV)
print("SHA-256:", digest)
print("Matches the export used in the manuscript:", digest == EXPECTED_SHA256)

In [ ]:
# Step 2: structure-type metadata (initial structures), pinned public commit
META_URL = ("https://raw.githubusercontent.com/UON-comp-chem/gp-representations/"
            "248802301bdef08cd03d5488dd270d0ac223bea2/data_generation/hydrogen-unrelaxed.db")
META_DB = WORK / "hydrogen-unrelaxed.db"

if META_DB.exists() and META_DB.stat().st_size > 70_000_000:
    print(f"Metadata database found: {META_DB} ({META_DB.stat().st_size/1e6:.1f} MB)")
else:
    with requests.get(META_URL, stream=True, timeout=120) as r:
        r.raise_for_status()
        size = int(r.headers.get("content-length", 0))
        done, next_report = 0, 10_000_000
        with open(META_DB, "wb") as fh:
            for chunk in r.iter_content(1 << 20):
                fh.write(chunk); done += len(chunk)
                if done >= next_report:
                    print(f"  downloaded {done/1e6:.0f} / {size/1e6:.0f} MB"); next_report += 10_000_000
    print(f"Saved {META_DB} ({META_DB.stat().st_size/1e6:.1f} MB)")

In [ ]:
# Step 3: audit functions (de-duplication, surfaces, structure type, prevalence, feasibility)
import ast
import collections as C
import csv
import json
import math
import re
import sqlite3
import sys
from pathlib import Path

DG_CORR = 0.24          # eV, Delta ZPE - T Delta S for H*, Norskov et al. 2005
DELTAS = (0.10, 0.15, 0.20, 0.30)


def per_h_energy(rec):
    e = float(rec["reaction_energy"])
    if rec["equation"] == "H2(g) + 2* -> 2H*":
        return e / 2.0
    if rec["equation"] == "0.5H2(g) + * -> H*":
        return e
    raise ValueError(rec["equation"])


def elements(formula):
    return sorted(set(re.findall(r"[A-Z][a-z]?", formula)))


def main(h_csv, udb, outdir):
    out = Path(outdir)
    out.mkdir(parents=True, exist_ok=True)
    csv.field_size_limit(10 ** 9)
    raw = list(csv.DictReader(open(h_csv)))
    rows, no_structure = [], 0
    for r in raw:
        systems = {x["name"]: x["aseId"] for x in json.loads(r["reactionSystems"])}
        if "Hstar" not in systems:          # no adsorbate structure stored: excluded
            no_structure += 1
            continue
        rows.append({"pub_id": "MamunHighT2019", "dft_functional": r["dftFunctional"],
                     "dft_code": r["dftCode"], "equation": r["Equation"], "coverages": r["coverages"],
                     "reaction_energy": r["reactionEnergy"], "hstar_id": systems["Hstar"],
                     "chemical_composition": r["chemicalComposition"],
                     "surface_composition": r["surfaceComposition"], "facet": r["facet"],
                     "sites": r["sites"]})
    audit = {"n_records": len(raw), "n_records_without_adsorbate_structure": no_structure,
             "n_records_used": len(rows)}
    audit["pub_id"] = dict(C.Counter(r["pub_id"] for r in rows))
    audit["functional"] = dict(C.Counter(r["dft_functional"] for r in rows))
    audit["code"] = dict(C.Counter(r["dft_code"] for r in rows))
    audit["equations"] = dict(C.Counter(r["equation"] for r in rows))
    audit["coverages"] = dict(C.Counter(r["coverages"] for r in rows))

    # de-duplicate on the relaxed adsorbate-structure id (last atoms id)
    uniq, dup_check = {}, []
    for r in rows:
        key = r["hstar_id"]
        e = per_h_energy(r)
        if key in uniq:
            dup_check.append(abs(uniq[key]["E"] - e))
            continue
        uniq[key] = {"surface": r["chemical_composition"], "slab": r["surface_composition"],
                     "facet": r["facet"], "site": json.loads(r["sites"])["H"], "E": e}
    audit["n_unique_configurations"] = len(uniq)
    audit["n_duplicates_removed"] = len(rows) - len(uniq)
    audit["max_abs_diff_duplicates_eV"] = max(dup_check) if dup_check else None

    # structure-type metadata from the unrelaxed database (read only)
    con = sqlite3.connect(f"file:{udb}?mode=ro", uri=True)
    meta = C.defaultdict(C.Counter)
    for (kvp,) in con.execute("select key_value_pairs from systems"):
        d = json.loads(kvp)
        meta[d["slab_name"]][(d.get("SB_symbol"), d.get("Pearson_symbol"), d.get("metalA"), d.get("metalB"))] += 1

    surf = C.defaultdict(list)
    for v in uniq.values():
        surf[(v["surface"], v["slab"], v["facet"])].append(v)

    table, n_meta_conflict, n_meta_missing, n_rule_mismatch = [], 0, 0, 0
    for (chem, slab, facet), confs in sorted(surf.items()):
        m = meta.get(slab)
        if not m:
            n_meta_missing += 1
            sb = pearson = mA = mB = None
        else:
            if len(m) > 1:
                n_meta_conflict += 1
            sb, pearson, mA, mB = m.most_common(1)[0][0]
        rule = "A1" if len(elements(slab)) == 1 else ("L12" if "3" in slab else "L10")
        if sb != rule:
            n_rule_mismatch += 1
        emin = min(c["E"] for c in confs)
        dg = emin + DG_CORR
        els = elements(slab)
        table.append({"surface": slab, "formula": chem, "facet": facet, "structure": sb,
                      "pearson": pearson, "metalA": mA, "metalB": mB if mB else "",
                      "system": "-".join(els), "n_sites": len(confs),
                      "n_unlabelled_sites": sum(1 for c in confs if "|" not in c["site"]),
                      "Emin_eV": round(emin, 6), "dGH_eV": round(dg, 6)})
    audit["n_surfaces"] = len(table)
    audit["n_surfaces_no_metadata"] = n_meta_missing
    audit["n_surfaces_metadata_conflict"] = n_meta_conflict
    audit["n_structure_rule_mismatch"] = n_rule_mismatch
    audit["structure_counts"] = dict(C.Counter(t["structure"] for t in table))
    audit["structure_by_facet"] = {f"{k[0]}|{k[1]}": v for k, v in
                                   C.Counter((t["structure"], t["facet"]) for t in table).items()}
    audit["n_chemical_systems"] = len({t["system"] for t in table})
    audit["n_elements"] = len({e for t in table for e in t["system"].split("-")})
    audit["sites_per_surface"] = dict(sorted(C.Counter(t["n_sites"] for t in table).items()))
    dgs = sorted(t["dGH_eV"] for t in table)
    q = lambda p: dgs[min(len(dgs) - 1, int(p * len(dgs)))]
    audit["dGH_quantiles_eV"] = {"min": dgs[0], "q10": q(.1), "q25": q(.25), "median": q(.5),
                                 "q75": q(.75), "q90": q(.9), "max": dgs[-1]}

    # feasibility: prevalence of in-window surfaces, overall and by structure type
    prev = {}
    for d in DELTAS:
        hit = [t for t in table if abs(t["dGH_eV"]) <= d]
        prev[f"{d:.2f}"] = {"n_hits": len(hit), "prevalence": round(len(hit) / len(table), 4),
                            "by_structure": dict(C.Counter(t["structure"] for t in hit)),
                            "n_systems_with_hit": len({t["system"] for t in hit})}
    audit["window"] = prev

    # attainability: conformal p-values are bounded below by 1/(n_cal+1); BH at level a
    # with m candidates can select k units only if 1/(n_cal+1) <= a*k/m, i.e.
    # k >= k_min = ceil(m / (a*(n_cal+1))). Compare k_min with the expected number of
    # in-window candidates in the pool (prevalence * m). Pools are the held-out structure
    # types; calibration sizes are the planned 30 % of the remaining surfaces.
    gate = {}
    struct_n = C.Counter(t["structure"] for t in table)
    for held in ("L10", "L12"):
        m = struct_n[held]
        n_cal = int(0.3 * (len(table) - m))
        for d in DELTAS:
            hits_pool = sum(1 for t in table if t["structure"] == held and abs(t["dGH_eV"]) <= d)
            for a in (0.1, 0.2, 0.3):
                kmin = math.ceil(m / (a * (n_cal + 1)))
                gate[f"pool={held},delta={d:.2f},alpha={a}"] = {
                    "m": m, "n_cal": n_cal, "k_min": kmin, "hits_in_pool": hits_pool,
                    "attainable": hits_pool >= kmin}
    audit["attainability_gate"] = gate
    audit["gate_all_attainable"] = all(v["attainable"] for v in gate.values())

    with open(out / "audit_surfaces.csv", "w", newline="") as fh:
        w = csv.DictWriter(fh, fieldnames=list(table[0].keys()))
        w.writeheader()
        w.writerows(table)
    with open(out / "audit_summary.json", "w") as fh:
        json.dump(audit, fh, indent=1)
    print(json.dumps(audit, indent=1))

print('Audit functions loaded.')

In [ ]:
# Step 4: run the audit; small result files are saved to Drive
print("Running audit ...")
main(str(EXPORT_CSV), str(META_DB), str(OUT_DIR))
print()
print("Saved:", OUT_DIR / "audit_surfaces.csv")
print("Saved:", OUT_DIR / "audit_summary.json")